In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

class ShallowNeuralHawkes(nn.Module):
    """
    Implementação do Shallow Neural Hawkes Process do paper:
    "Shallow Neural Hawkes: Non-parametric kernel estimation for Hawkes processes"
    
    Ideia principal: Usar rede neural rasa para aprender kernels φ(t) não-paramétricos
    ao invés de assumir φ(t) = α×exp(-β×t)
    """
    
    def __init__(self, num_types=1, hidden_dim=64, max_time_delta=10.0, 
                 num_basis=50, activation='relu'):
        """
        Args:
            num_types: número de tipos de eventos (1=univariado, 2=bivariado, etc)
            hidden_dim: dimensão da camada oculta
            max_time_delta: máximo Δt considerado para os kernels
            num_basis: número de pontos base para discretização dos kernels
            activation: função de ativação ('relu', 'tanh', 'sigmoid')
        """
        super(ShallowNeuralHawkes, self).__init__()
        
        self.num_types = num_types
        self.hidden_dim = hidden_dim
        self.max_time_delta = max_time_delta
        self.num_basis = num_basis
        
        print(f"🧠 Inicializando Shallow Neural Hawkes")
        print(f"   • Tipos de eventos: {num_types}")
        print(f"   • Hidden dim: {hidden_dim}")
        print(f"   • Max Δt: {max_time_delta}")
        print(f"   • Basis functions: {num_basis}")
        
        # Baseline rates μ (learnable parameters)
        self.mu = nn.Parameter(torch.ones(num_types) * 0.1)
        
        # Pontos temporais para discretização dos kernels
        self.register_buffer('time_points', 
                           torch.linspace(0, max_time_delta, num_basis))
        
        # Redes neurais para cada kernel φᵢⱼ(t)
        # Para cada par (i,j): eventos tipo j → excitação tipo i
        self.kernel_networks = nn.ModuleDict()
        
        for i in range(num_types):
            for j in range(num_types):
                # Rede rasa: input_dim=1 (Δt) → hidden → output_dim=1 (φ(Δt))
                network = nn.Sequential(
                    nn.Linear(1, hidden_dim),
                    self._get_activation(activation),
                    nn.Linear(hidden_dim, hidden_dim),
                    self._get_activation(activation),
                    nn.Linear(hidden_dim, 1),
                    nn.Softplus()  # Garantir φ(t) ≥ 0
                )
                self.kernel_networks[f'{i}_{j}'] = network
        
        print(f"   • Kernels criados: {list(self.kernel_networks.keys())}")
    
    def _get_activation(self, activation):
        """Retorna função de ativação"""
        if activation == 'relu':
            return nn.ReLU()
        elif activation == 'tanh':
            return nn.Tanh()
        elif activation == 'sigmoid':
            return nn.Sigmoid()
        else:
            return nn.ReLU()
    
    def compute_kernel_value(self, delta_t, from_type, to_type):
        """
        Computa φᵢⱼ(Δt) usando a rede neural
        
        Args:
            delta_t: diferença temporal (t - tᵢ)
            from_type: tipo do evento causador (j)  
            to_type: tipo do evento afetado (i)
            
        Returns:
            kernel_value: φᵢⱼ(Δt)
        """
        if isinstance(delta_t, (int, float)):
            delta_t = torch.tensor([[delta_t]], dtype=torch.float32)
        elif len(delta_t.shape) == 1:
            delta_t = delta_t.unsqueeze(-1)
            
        network_key = f'{to_type}_{from_type}'
        kernel_net = self.kernel_networks[network_key]
        
        return kernel_net(delta_t).squeeze(-1)
    
    def compute_intensity(self, t, event_history, event_types, target_type):
        """
        Computa intensidade λᵢ(t) para tipo específico
        
        λᵢ(t) = μᵢ + Σⱼ Σₖ φᵢⱼ(t - tₖ)
        onde tₖ são eventos do tipo j que ocorreram antes de t
        
        Args:
            t: tempo atual
            event_history: lista de timestamps dos eventos
            event_types: lista de tipos correspondentes aos eventos
            target_type: tipo i para o qual calcular λᵢ(t)
            
        Returns:
            intensity: λᵢ(t)
        """
        # Baseline
        intensity = self.mu[target_type]
        
        # Somar contribuições de todos eventos anteriores
        for event_time, event_type in zip(event_history, event_types):
            if event_time < t:
                delta_t = t - event_time
                
                # Só considerar eventos dentro do horizonte temporal
                if delta_t <= self.max_time_delta:
                    kernel_value = self.compute_kernel_value(
                        delta_t, from_type=event_type, to_type=target_type
                    )
                    intensity += kernel_value
        
        return intensity
    
    def compute_all_intensities(self, t, event_history, event_types):
        """
        Computa λᵢ(t) para todos os tipos simultaneamente
        """
        intensities = []
        for target_type in range(self.num_types):
            intensity = self.compute_intensity(t, event_history, event_types, target_type)
            intensities.append(intensity)
        
        return torch.stack(intensities)
    
    def log_likelihood(self, event_sequences):
        """
        Computa log-likelihood dos dados observados
        
        LL = Σᵢ [log λᵢ(tᵢ)] - Σᵢ ∫₀ᵀ λᵢ(s) ds
           = Σᵢ [log λᵢ(tᵢ)] - Σᵢ [integral_part_i]
        
        Args:
            event_sequences: lista de sequências [(t₁,type₁), (t₂,type₂), ...]
            
        Returns:
            log_likelihood: LL total
        """
        total_ll = 0.0
        
        for sequence in event_sequences:
            if len(sequence) == 0:
                continue
                
            times = [event[0] for event in sequence]
            types = [event[1] for event in sequence]
            T_max = max(times)
            
            # Termo 1: Σ log λᵢ(tᵢ)
            log_intensity_sum = 0.0
            
            for i, (t_i, type_i) in enumerate(sequence):
                # Histórico até evento i (não incluindo i)
                history_times = times[:i]
                history_types = types[:i]
                
                # Calcular λ(tᵢ)
                intensity = self.compute_intensity(t_i, history_times, history_types, type_i)
                
                # Evitar log(0)
                intensity = torch.clamp(intensity, min=1e-8)
                log_intensity_sum += torch.log(intensity)
            
            # Termo 2: - ∫₀ᵀ λᵢ(s) ds (aproximação por quadratura)
            integral_penalty = self._compute_integral_penalty(times, types, T_max)
            
            sequence_ll = log_intensity_sum - integral_penalty
            total_ll += sequence_ll
        
        return total_ll
    
    def _compute_integral_penalty(self, times, types, T_max, num_quadrature=100):
        """
        Aproxima ∫₀ᵀ λᵢ(s) ds usando quadratura
        """
        dt = T_max / num_quadrature
        quadrature_points = torch.linspace(0, T_max, num_quadrature)
        
        total_integral = 0.0
        
        for t_quad in quadrature_points:
            # Para cada ponto de quadratura, calcular intensidades
            for target_type in range(self.num_types):
                intensity = self.compute_intensity(t_quad, times, types, target_type)
                total_integral += intensity * dt
        
        return total_integral
    
    def get_learned_kernels(self, resolution=100):
        """
        Extrai os kernels aprendidos φᵢⱼ(t) para visualização
        
        Returns:
            dict: {(i,j): (time_points, kernel_values)}
        """
        time_grid = torch.linspace(0, self.max_time_delta, resolution).unsqueeze(-1)
        learned_kernels = {}
        
        with torch.no_grad():
            for i in range(self.num_types):
                for j in range(self.num_types):
                    kernel_values = self.compute_kernel_value(time_grid, from_type=j, to_type=i)
                    learned_kernels[(i, j)] = (time_grid.squeeze().numpy(), 
                                             kernel_values.numpy())
        
        return learned_kernels
    
    def simulate_events(self, T_max=10.0, max_events=1000):
        """
        Simula eventos usando o modelo aprendido (Thinning Algorithm)
        """
        events = []
        t = 0.0
        
        while t < T_max and len(events) < max_events:
            # Calcular intensidade máxima (upper bound)
            with torch.no_grad():
                times = [e[0] for e in events]
                types = [e[1] for e in events]
                intensities = self.compute_all_intensities(t, times, types)
                lambda_max = torch.sum(intensities) * 1.5  # safety factor
            
            # Propor próximo evento
            if lambda_max > 0:
                dt = np.random.exponential(1.0 / lambda_max.item())
                t_proposed = t + dt
                
                if t_proposed < T_max:
                    # Calcular intensidade real no tempo proposto
                    with torch.no_grad():
                        intensities_real = self.compute_all_intensities(t_proposed, times, types)
                        lambda_total_real = torch.sum(intensities_real)
                    
                    # Teste de aceitação
                    if np.random.random() < (lambda_total_real / lambda_max).item():
                        # Aceitar evento - determinar tipo
                        probs = intensities_real / lambda_total_real
                        event_type = np.random.choice(self.num_types, p=probs.numpy())
                        events.append((t_proposed, event_type))
                    
                    t = t_proposed
                else:
                    break
            else:
                t += 0.1  # small step if no intensity
        
        return events

class HawkesDataset(Dataset):
    """Dataset para treinar Shallow Neural Hawkes"""
    
    def __init__(self, event_sequences):
        self.sequences = event_sequences
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx]

def generate_synthetic_hawkes_data(num_sequences=50, T_max=20.0, 
                                 mu=0.1, alpha=0.5, beta=1.0, num_types=1):
    """
    Gera dados sintéticos de Hawkes Process para teste
    """
    print(f"📊 Gerando {num_sequences} sequências sintéticas...")
    
    sequences = []
    
    for seq_idx in range(num_sequences):
        events = []
        t = 0.0
        
        # Simular sequência usando Hawkes clássico
        while t < T_max:
            # Calcular intensidade atual
            intensity = mu
            for event_time, event_type in events:
                if t > event_time:
                    intensity += alpha * np.exp(-beta * (t - event_time))
            
            # Próximo evento
            if intensity > 0:
                dt = np.random.exponential(1.0 / intensity)
                t += dt
                
                if t < T_max:
                    event_type = 0 if num_types == 1 else np.random.randint(num_types)
                    events.append((t, event_type))
            else:
                t += 0.1
        
        sequences.append(events)
    
    total_events = sum(len(seq) for seq in sequences)
    print(f"✅ Geradas {len(sequences)} sequências com {total_events} eventos totais")
    
    return sequences

def train_shallow_neural_hawkes(model, train_sequences, num_epochs=100, lr=0.001):
    """
    Treina o modelo Shallow Neural Hawkes
    """
    print(f"\n🚀 Iniciando treinamento...")
    print(f"   • Épocas: {num_epochs}")
    print(f"   • Learning rate: {lr}")
    print(f"   • Sequências de treino: {len(train_sequences)}")
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        
        # Calcular negative log-likelihood
        nll = -model.log_likelihood(train_sequences)
        
        # Backpropagation
        nll.backward()
        
        # Gradient clipping para estabilidade
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        loss_history.append(nll.item())
        
        if (epoch + 1) % 20 == 0:
            print(f"   Época {epoch+1}/{num_epochs}: Loss = {nll.item():.4f}")
    
    print(f"✅ Treinamento concluído!")
    return loss_history

def plot_learned_kernels(model, title="Kernels Aprendidos"):
    """
    Visualiza os kernels φᵢⱼ(t) aprendidos
    """
    kernels = model.get_learned_kernels()
    
    if model.num_types == 1:
        # Hawkes univariado
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        time_points, kernel_values = kernels[(0, 0)]
        
        ax.plot(time_points, kernel_values, 'b-', linewidth=2, label='φ(t)')
        
        # Comparar com kernel exponencial teórico (se conhecido)
        theoretical = 0.5 * np.exp(-1.0 * time_points)  # α=0.5, β=1.0
        ax.plot(time_points, theoretical, 'r--', linewidth=2, label='Teórico: 0.5×exp(-t)')
        
        ax.set_xlabel('Δt')
        ax.set_ylabel('φ(Δt)')
        ax.set_title(f'{title} - Hawkes Univariado')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
    else:
        # Hawkes multivariado
        fig, axes = plt.subplots(model.num_types, model.num_types, 
                               figsize=(4*model.num_types, 4*model.num_types))
        
        if model.num_types == 1:
            axes = [[axes]]
        elif model.num_types == 2:
            if len(axes.shape) == 1:
                axes = axes.reshape(2, 1)
        
        for i in range(model.num_types):
            for j in range(model.num_types):
                ax = axes[i][j] if model.num_types > 1 else axes[0][0]
                time_points, kernel_values = kernels[(i, j)]
                
                ax.plot(time_points, kernel_values, 'b-', linewidth=2)
                ax.set_title(f'φ_{i+1},{j+1}(t): Tipo {j+1} → Tipo {i+1}')
                ax.set_xlabel('Δt')
                ax.set_ylabel('φ(Δt)')
                ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def demonstrate_shallow_neural_hawkes():
    """
    Demonstração completa do Shallow Neural Hawkes
    """
    print("=" * 60)
    print("🎯 DEMONSTRAÇÃO: SHALLOW NEURAL HAWKES")
    print("=" * 60)
    
    # 1. Gerar dados sintéticos
    train_data = generate_synthetic_hawkes_data(
        num_sequences=30, T_max=15.0, 
        mu=0.1, alpha=0.5, beta=1.0, num_types=1
    )
    
    # 2. Criar modelo
    model = ShallowNeuralHawkes(
        num_types=1, 
        hidden_dim=32,
        max_time_delta=5.0,
        num_basis=50
    )
    
    print(f"\n📋 Parâmetros iniciais:")
    print(f"   • μ (baseline): {model.mu.data.numpy()}")
    
    # 3. Treinar modelo
    loss_history = train_shallow_neural_hawkes(
        model, train_data, num_epochs=200, lr=0.005
    )
    
    # 4. Resultados
    print(f"\n📋 Parâmetros aprendidos:")
    print(f"   • μ (baseline): {model.mu.data.numpy()}")
    
    # 5. Visualizar kernels aprendidos
    plot_learned_kernels(model, "Kernels Aprendidos vs Teórico")
    
    # 6. Simular novos eventos
    print(f"\n🎲 Simulando novos eventos...")
    simulated_events = model.simulate_events(T_max=10.0)
    print(f"   • Eventos simulados: {len(simulated_events)}")
    if len(simulated_events) > 0:
        print(f"   • Primeiros 5 eventos: {simulated_events[:5]}")
    
    # 7. Plot da loss
    plt.figure(figsize=(10, 6))
    plt.plot(loss_history, 'b-', linewidth=2)
    plt.xlabel('Época')
    plt.ylabel('Negative Log-Likelihood')
    plt.title('Convergência do Treinamento')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return model, train_data, loss_history

if __name__ == "__main__":
    # Executar demonstração
    model, data, losses = demonstrate_shallow_neural_hawkes()
    
    print(f"\n💡 RESUMO:")
    print(f"✅ Modelo treinado com sucesso")
    print(f"✅ Kernels não-paramétricos aprendidos")
    print(f"✅ Capacidade de simulação verificada")
    print(f"\n🎯 O modelo aprendeu a aproximar φ(t) = α×exp(-β×t) usando redes neurais!")

🎯 DEMONSTRAÇÃO: SHALLOW NEURAL HAWKES
📊 Gerando 30 sequências sintéticas...
✅ Geradas 30 sequências com 66 eventos totais
🧠 Inicializando Shallow Neural Hawkes
   • Tipos de eventos: 1
   • Hidden dim: 32
   • Max Δt: 5.0
   • Basis functions: 50
   • Kernels criados: ['0_0']

📋 Parâmetros iniciais:
   • μ (baseline): [0.1]

🚀 Iniciando treinamento...
   • Épocas: 200
   • Learning rate: 0.005
   • Sequências de treino: 30


RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation.